In [1]:
pip install kafka-python faker

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import random
import time
import uuid
import threading
from datetime import datetime, timedelta, timezone
from faker import Faker
from kafka import KafkaProducer
from kafka.errors import KafkaError

In [3]:

# ═══════════════════════════════════════════════════════════════════════════════
#  ██████╗ ██████╗ ███╗   ██╗████████╗██████╗  ██████╗ ██╗
# ██╔════╝██╔═══██╗████╗  ██║╚══██╔══╝██╔══██╗██╔═══██╗██║
# ██║     ██║   ██║██╔██╗ ██║   ██║   ██████╔╝██║   ██║██║
# ██║     ██║   ██║██║╚██╗██║   ██║   ██╔══██╗██║   ██║██║
# ╚██████╗╚██████╔╝██║ ╚████║   ██║   ██║  ██║╚██████╔╝███████╗
#  ╚═════╝ ╚═════╝ ╚═╝  ╚═══╝   ╚═╝   ╚═╝  ╚═╝ ╚═════╝ ╚══════╝
#  PANEL — Edit these values before running
# ═══════════════════════════════════════════════════════════════════════════════

# ── Kafka Connection ───────────────────────────────────────────────────────────
BOOTSTRAP_SERVERS = "localhost:9092"

# ── Topic ON/OFF Switches ──────────────────────────────────────────────────────
# Set True to enable a topic producer, False to skip it entirely
ENABLE_CLICKSTREAM   = True
ENABLE_TELEMETRY     = True
ENABLE_ORDER_EVENTS  = True
ENABLE_DQ_ALERTS     = True

# ── Event Count Limits (per topic) ────────────────────────────────────────────
# How many events to produce before stopping that topic's producer.
# Set to 0 for UNLIMITED (runs until you press Ctrl+C)
CLICKSTREAM_MAX_EVENTS   = 500     # 0 = unlimited
TELEMETRY_MAX_EVENTS     = 300     # 0 = unlimited
ORDER_EVENTS_MAX_EVENTS  = 200     # 0 = unlimited
DQ_ALERTS_MAX_EVENTS     = 100     # 0 = unlimited

# ── Timing (seconds between events per topic) ─────────────────────────────────
# Each topic fires at a random interval between MIN and MAX seconds
CLICKSTREAM_INTERVAL_MIN   = 0.2   # seconds
CLICKSTREAM_INTERVAL_MAX   = 1.0

TELEMETRY_INTERVAL_MIN     = 0.5
TELEMETRY_INTERVAL_MAX     = 2.0

ORDER_INTERVAL_MIN         = 1.0
ORDER_INTERVAL_MAX         = 4.0

DQ_ALERTS_INTERVAL_MIN     = 2.0
DQ_ALERTS_INTERVAL_MAX     = 6.0

# ── Data Realism Toggles ──────────────────────────────────────────────────────
INJECT_LATE_ARRIVALS  = True    # 8% of clickstream events have backdated timestamps
INJECT_POISON_PILLS   = True    # 2% of events have bad/corrupt data (tests DQ)
ANONYMOUS_USER_RATE   = 0.30    # 30% of clickstream sessions have no user_id

# ═══════════════════════════════════════════════════════════════════════════════
#  END OF CONTROL PANEL
# ═══════════════════════════════════════════════════════════════════════════════

In [4]:

# ─── Shared Setup ─────────────────────────────────────────────────────────────
fake = Faker("en_US")
Faker.seed(0)
NOW = lambda: datetime.now(timezone.utc)

# Reference data pools
PRODUCT_IDS    = [f"PRD-{i:06d}" for i in range(1, 5001)]
CUSTOMER_IDS   = [f"CUST-{i:07d}" for i in range(1, 10001)]
WAREHOUSE_IDS  = [f"WH-{i:03d}" for i in range(1, 21)]
SUPPLIER_IDS   = [f"SUP-{i:05d}" for i in range(1, 501)]
CATEGORIES     = ["Electronics", "Clothing", "Home & Garden", "Sports",
                  "Books", "Food & Beverage", "Automotive", "Beauty", "Toys", "Office"]
COUNTRIES      = ["US", "US", "US", "UK", "CA", "DE", "AU", "JP", "SG", "BR"]
DEVICE_TYPES   = ["mobile", "mobile", "desktop", "desktop", "tablet"]
CARRIERS       = ["FedEx", "UPS", "DHL", "USPS", "Amazon Logistics"]
PAYMENT_METHODS = ["credit_card", "debit_card", "paypal", "bank_transfer"]
ORDER_STATUSES  = ["placed", "confirmed", "processing", "shipped", "delivered", "cancelled"]
DQ_SEVERITY     = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
DQ_RULE_TYPES   = ["null_check", "range_check", "referential_integrity",
                   "schema_drift", "duplicate_key", "type_mismatch", "freshness_check"]
DQ_TABLES       = ["bronze.clickstream_raw", "bronze.orders_raw",
                   "silver.customers", "silver.order_lines",
                   "bronze.device_telemetry", "silver.payments"]


In [5]:
# Active session pool (maintains session continuity for clickstream)
_sessions = {}

# ═══════════════════════════════════════════════════════════════════════════════
# EVENT BUILDERS
# ═════════════════════════════════════════════════════════

In [6]:
def build_clickstream_event():
    """Page views, searches, cart actions, purchases."""
    now = NOW()
    event_types = (["page_view"] * 4 + ["search"] * 2 + ["click"] * 2
                   + ["add_to_cart", "remove_from_cart", "purchase"])

    # Session management
    user_id = None if random.random() < ANONYMOUS_USER_RATE else random.choice(CUSTOMER_IDS)
    session_key = user_id or f"anon-{random.randint(1,500)}"

    if session_key not in _sessions:
        _sessions[session_key] = {
            "session_id":   str(uuid.uuid4()),
            "page_count":   0,
            "cart_value":   0.0,
            "cart_items":   [],
            "device":       random.choice(DEVICE_TYPES),
            "country":      random.choice(COUNTRIES),
            "city":         fake.city(),
            "ip":           fake.ipv4(),
            "ua":           fake.user_agent(),
        }
        # Expire old sessions occasionally
        if len(_sessions) > 200:
            oldest = list(_sessions.keys())[0]
            del _sessions[oldest]

    s = _sessions[session_key]
    s["page_count"] += 1
    event_type = random.choice(event_types)

    product_id, category, search_query = None, None, None
    purchase_amount, cart_value = None, None

    if event_type in ("add_to_cart", "click", "page_view"):
        product_id = random.choice(PRODUCT_IDS)
        category   = random.choice(CATEGORIES)
        if event_type == "add_to_cart":
            price = round(random.uniform(5, 500), 2)
            s["cart_items"].append(product_id)
            s["cart_value"] = round(s["cart_value"] + price, 2)
            cart_value = s["cart_value"]
    elif event_type == "remove_from_cart" and s["cart_items"]:
        product_id    = s["cart_items"].pop()
        s["cart_value"] = max(0.0, round(s["cart_value"] - random.uniform(5, 100), 2))
        cart_value    = s["cart_value"]
    elif event_type == "purchase":
        purchase_amount = s["cart_value"] if s["cart_value"] > 0 else round(random.uniform(10, 500), 2)
        cart_value      = purchase_amount
        s["cart_items"] = []
        s["cart_value"] = 0.0
    elif event_type == "search":
        search_query = f"{fake.word()} {random.choice(CATEGORIES).lower()}"

    # Late arrival injection
    is_late = INJECT_LATE_ARRIVALS and random.random() < 0.08
    event_time = (now - timedelta(minutes=random.randint(15, 90))) if is_late else now

    # Poison pill injection
    is_poison = INJECT_POISON_PILLS and random.random() < 0.02

    event = {
        "event_id":          str(uuid.uuid4()),
        "event_type":        event_type,
        "session_id":        s["session_id"],
        "user_id":           user_id,
        "device_type":       s["device"],
        "event_time":        event_time.isoformat(),
        "processing_time":   now.isoformat(),
        "page_url":          f"/{event_type.replace('_', '-')}",
        "referrer_url":      f"https://{fake.domain_name()}" if random.random() < 0.3 else None,
        "product_id":        product_id,
        "category":          category,
        "search_query":      search_query,
        "cart_value":        cart_value,
        "purchase_amount":   purchase_amount,
        "ip_address":        s["ip"],
        "user_agent":        s["ua"],
        "geo_country":       s["country"],
        "geo_city":          s["city"],
        "session_page_count": s["page_count"],
        "_is_late_arrival":  is_late,
        "_is_poison_pill":   is_poison,
    }

    # Corrupt if poison pill
    if is_poison:
        pill = random.choice(["null_event_id", "bad_amount", "future_time", "wrong_type"])
        if pill == "null_event_id":      event["event_id"]      = None
        elif pill == "bad_amount":       event["purchase_amount"] = "NOT_A_NUMBER"
        elif pill == "future_time":      event["event_time"]    = (now + timedelta(hours=25)).isoformat()
        elif pill == "wrong_type":       event["session_page_count"] = "many"

    return event, s["session_id"]  # key = session_id for partition affinity


In [7]:
def build_telemetry_event():
    """IoT / device sensor readings — temperature, CPU, memory, network."""
    now = NOW()
    device_id   = f"DEV-{random.randint(1, 500):05d}"
    device_type = random.choice(["router", "sensor", "gateway", "edge_node", "pos_terminal"])

    is_late   = INJECT_LATE_ARRIVALS and random.random() < 0.06
    is_poison = INJECT_POISON_PILLS and random.random() < 0.02

    event_time = (now - timedelta(minutes=random.randint(10, 60))) if is_late else now

    event = {
        "telemetry_id":      str(uuid.uuid4()),
        "device_id":         device_id,
        "device_type":       device_type,
        "firmware_version":  f"v{random.randint(1,4)}.{random.randint(0,9)}.{random.randint(0,20)}",
        "location_id":       random.choice(WAREHOUSE_IDS),
        "event_time":        event_time.isoformat(),
        "processing_time":   now.isoformat(),
        "metrics": {
            "cpu_pct":           round(random.uniform(0, 100), 2),
            "memory_pct":        round(random.uniform(10, 95), 2),
            "disk_pct":          round(random.uniform(5, 90), 2),
            "temperature_c":     round(random.uniform(20, 80), 1),
            "network_in_kbps":   round(random.uniform(0, 1000), 1),
            "network_out_kbps":  round(random.uniform(0, 500), 1),
            "error_count":       random.choices([0, 0, 0, 1, 2, 5], weights=[70,10,5,8,5,2])[0],
            "uptime_seconds":    random.randint(60, 86400 * 30),
        },
        "status":            random.choices(
                                 ["online", "online", "online", "degraded", "offline"],
                                 weights=[80, 5, 5, 7, 3])[0],
        "alert_triggered":   random.random() < 0.05,
        "_is_late_arrival":  is_late,
        "_is_poison_pill":   is_poison,
    }

    if is_poison:
        pill = random.choice(["null_device", "negative_cpu", "wrong_type"])
        if pill == "null_device":    event["device_id"]              = None
        elif pill == "negative_cpu": event["metrics"]["cpu_pct"]     = -999
        elif pill == "wrong_type":   event["metrics"]["temperature_c"] = "HOT"

    return event, device_id  # key = device_id for partition affinity


In [8]:

def build_order_event():
    """Order lifecycle events — placed, updated, shipped, delivered, cancelled."""
    now = NOW()
    order_id    = f"ORD-{random.randint(1, 100000):08d}"
    customer_id = random.choice(CUSTOMER_IDS)
    event_type  = random.choices(
        ["order_placed", "order_confirmed", "order_shipped",
         "order_delivered", "order_cancelled", "order_updated"],
        weights=[30, 20, 20, 15, 10, 5]
    )[0]

    is_poison = INJECT_POISON_PILLS and random.random() < 0.02

    n_items = random.randint(1, 5)
    line_items = [
        {
            "product_id":  random.choice(PRODUCT_IDS),
            "quantity":    random.randint(1, 10),
            "unit_price":  round(random.uniform(2, 800), 2),
            "discount_pct": random.choices([0, 5, 10, 15, 20], weights=[60, 15, 12, 8, 5])[0],
        }
        for _ in range(n_items)
    ]
    total = round(sum(i["unit_price"] * i["quantity"] for i in line_items), 2)

    event = {
        "event_id":          str(uuid.uuid4()),
        "event_type":        event_type,
        "order_id":          order_id,
        "customer_id":       customer_id,
        "warehouse_id":      random.choice(WAREHOUSE_IDS),
        "event_time":        now.isoformat(),
        "order_date":        now.strftime("%Y-%m-%d"),
        "status":            event_type.replace("order_", ""),
        "channel":           random.choice(["web", "mobile_app", "marketplace", "in_store"]),
        "currency":          random.choice(["USD", "EUR", "GBP"]),
        "total_amount":      total,
        "discount_amount":   round(total * random.uniform(0, 0.15), 2),
        "tax_amount":        round(total * 0.08, 2),
        "shipping_amount":   round(random.uniform(0, 25), 2),
        "line_items":        line_items,
        "shipping_country":  random.choice(COUNTRIES),
        "carrier":           random.choice(CARRIERS) if event_type == "order_shipped" else None,
        "tracking_number":   fake.uuid4()[:16].upper() if event_type == "order_shipped" else None,
        "cancellation_reason": random.choice(["changed_mind", "found_cheaper",
                                              "duplicate_order", "item_unavailable"])
                               if event_type == "order_cancelled" else None,
        "_is_poison_pill":   is_poison,
    }

    if is_poison:
        pill = random.choice(["null_order_id", "negative_total", "null_customer"])
        if pill == "null_order_id":   event["order_id"]     = None
        elif pill == "negative_total": event["total_amount"] = -abs(total)
        elif pill == "null_customer":  event["customer_id"]  = None

    return event, order_id  # key = order_id for ordering guarantees



In [9]:
def build_dq_alert_event():
    """Data quality alert notifications from DLT pipeline expectations."""
    now     = NOW()
    rule    = random.choice(DQ_RULE_TYPES)
    table   = random.choice(DQ_TABLES)
    sev     = random.choices(DQ_SEVERITY, weights=[40, 35, 18, 7])[0]
    fail_ct = random.randint(1, 500)
    total   = random.randint(fail_ct, 10000)

    event = {
        "alert_id":           str(uuid.uuid4()),
        "alert_time":         now.isoformat(),
        "severity":           sev,
        "rule_type":          rule,
        "rule_name":          f"{rule}__{table.split('.')[-1]}",
        "pipeline_name":      random.choice(["supply_chain_dlt", "clickstream_dlt",
                                              "finance_batch", "telemetry_stream"]),
        "table_name":         table,
        "failed_record_count": fail_ct,
        "total_record_count":  total,
        "failure_rate_pct":   round((fail_ct / total) * 100, 4),
        "sample_failed_values": [
            str(fake.uuid4()[:8]) for _ in range(min(3, fail_ct))
        ],
        "action_taken":       random.choice(["quarantined", "dropped", "warned", "failed_pipeline"]),
        "resolution_required": sev in ("HIGH", "CRITICAL"),
        "runbook_url":        f"https://wiki.omnioc.internal/dq/{rule}",
        "correlation_id":     str(uuid.uuid4()),
    }
    return event, table  # key = table name


In [10]:
# ═══════════════════════════════════════════════════════════════════════════════
# KAFKA PRODUCER FACTORY
# ═══════════════════════════════════════════════════════════════════════════════

def make_producer():
    return KafkaProducer(
        bootstrap_servers=BOOTSTRAP_SERVERS,
        value_serializer=lambda v: json.dumps(v, default=str).encode("utf-8"),
        key_serializer=lambda k: k.encode("utf-8") if k else None,
        acks="all",
        retries=3,
        linger_ms=5,
    )

In [11]:
# ═══════════════════════════════════════════════════════════════════════════════
# TOPIC WORKER THREADS
# ═══════════════════════════════════════════════════════════════════════════════

def run_topic(name, topic, builder_fn, max_events, interval_min, interval_max,
              producer, stop_event, counters, lock):
    """
    Generic worker that runs in its own thread.
    Produces events until max_events reached (or 0 = unlimited).
    """
    sent = 0
    label = f"[{name:<18}]"

    while not stop_event.is_set():
        if max_events > 0 and sent >= max_events:
            with lock:
                print(f"{label} ✅ Reached limit of {max_events} events. Done.")
            break

        try:
            event, key = builder_fn()
            future = producer.send(topic, key=key, value=event)
            future.get(timeout=10)   # Block briefly to catch errors early

            sent += 1
            pill  = "💊 " if event.get("_is_poison_pill") else ""
            late  = "🕐 " if event.get("_is_late_arrival") else ""
            etype = event.get("event_type") or event.get("alert_id", "")[:8] or name.lower()

            with lock:
                counters[name] = sent
                remaining = f"  ({max_events - sent} left)" if max_events > 0 else ""
                print(f"{label} #{sent:>4} {pill}{late}→ {etype:<22} key={str(key)[:20]:<20}{remaining}")

        except KafkaError as e:
            with lock:
                print(f"{label} ❌ Kafka error: {e}")

        # Random sleep between events
        sleep_time = random.uniform(interval_min, interval_max)
        stop_event.wait(timeout=sleep_time)

    with lock:
        print(f"{label} 🏁 Thread finished. Total sent: {sent}")



In [12]:

# ═══════════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════════

def main():
    print()
    print("╔══════════════════════════════════════════════════════════════╗")
    print("║       OmniOC Multi-Topic Kafka Producer (PyCharm)           ║")
    print(f"║       Broker: {BOOTSTRAP_SERVERS:<47}║")
    print("╠══════════════════════════════════════════════════════════════╣")

    active = []
    if ENABLE_CLICKSTREAM:
        lim = f"{CLICKSTREAM_MAX_EVENTS} events" if CLICKSTREAM_MAX_EVENTS > 0 else "unlimited"
        print(f"║  ✅ clickstream-events   │ {lim:<34}║")
        active.append("clickstream")
    if ENABLE_TELEMETRY:
        lim = f"{TELEMETRY_MAX_EVENTS} events" if TELEMETRY_MAX_EVENTS > 0 else "unlimited"
        print(f"║  ✅ device-telemetry     │ {lim:<34}║")
        active.append("telemetry")
    if ENABLE_ORDER_EVENTS:
        lim = f"{ORDER_EVENTS_MAX_EVENTS} events" if ORDER_EVENTS_MAX_EVENTS > 0 else "unlimited"
        print(f"║  ✅ order-events         │ {lim:<34}║")
        active.append("orders")
    if ENABLE_DQ_ALERTS:
        lim = f"{DQ_ALERTS_MAX_EVENTS} events" if DQ_ALERTS_MAX_EVENTS > 0 else "unlimited"
        print(f"║  ✅ dq-alerts            │ {lim:<34}║")
        active.append("dq")

    if not active:
        print("║  ⚠️  No topics enabled! Set ENABLE_* = True in control panel ║")
        print("╚══════════════════════════════════════════════════════════════╝")
        return

    print(f"║  💊 Poison pills: {'ON' if INJECT_POISON_PILLS else 'OFF'}  │  🕐 Late arrivals: {'ON' if INJECT_LATE_ARRIVALS else 'OFF':<22}║")
    print("╚══════════════════════════════════════════════════════════════╝")
    print("  Press Ctrl+C to stop at any time\n")

    # Connect to Kafka
    try:
        producer = make_producer()
        print(f"  🔌 Connected to Kafka at {BOOTSTRAP_SERVERS}\n")
    except Exception as e:
        print(f"  ❌ Cannot connect to Kafka: {e}")
        print("  👉 Make sure Docker Kafka is running:")
        print("     docker compose -f C:/omnioc/docker/docker-compose.yml start")
        return

    stop_event = threading.Event()
    counters   = {}
    lock       = threading.Lock()
    threads    = []

    # ── Launch one thread per enabled topic ──────────────────────────────────
    configs = [
        (ENABLE_CLICKSTREAM,  "Clickstream",    "clickstream-events",
         build_clickstream_event, CLICKSTREAM_MAX_EVENTS,
         CLICKSTREAM_INTERVAL_MIN, CLICKSTREAM_INTERVAL_MAX),

        (ENABLE_TELEMETRY,    "Telemetry",      "device-telemetry",
         build_telemetry_event,   TELEMETRY_MAX_EVENTS,
         TELEMETRY_INTERVAL_MIN,  TELEMETRY_INTERVAL_MAX),

        (ENABLE_ORDER_EVENTS, "Order Events",   "order-events",
         build_order_event,       ORDER_EVENTS_MAX_EVENTS,
         ORDER_INTERVAL_MIN,      ORDER_INTERVAL_MAX),

        (ENABLE_DQ_ALERTS,    "DQ Alerts",      "dq-alerts",
         build_dq_alert_event,    DQ_ALERTS_MAX_EVENTS,
         DQ_ALERTS_INTERVAL_MIN,  DQ_ALERTS_INTERVAL_MAX),
    ]

    for enabled, name, topic, builder, max_ev, imin, imax in configs:
        if not enabled:
            continue
        t = threading.Thread(
            target=run_topic,
            args=(name, topic, builder, max_ev, imin, imax,
                  producer, stop_event, counters, lock),
            daemon=True,
            name=f"Producer-{name}",
        )
        threads.append((name, t))
        t.start()

    # ── Wait for all threads to finish or Ctrl+C ─────────────────────────────
    try:
        while any(t.is_alive() for _, t in threads):
            time.sleep(0.5)

        # All threads finished naturally (all hit their limits)
        stop_event.set()

    except KeyboardInterrupt:
        print("\n\n  🛑 Ctrl+C received — stopping all producers...\n")
        stop_event.set()

    finally:
        for _, t in threads:
            t.join(timeout=5)

        producer.flush()
        producer.close()

        # ── Final summary ─────────────────────────────────────────────────────
        print()
        print("╔══════════════════════════════════════════════════════════════╗")
        print("║                    📊 Final Summary                         ║")
        print("╠══════════════════════════════════════════════════════════════╣")
        total = 0
        for name, _ in threads:
            count = counters.get(name, 0)
            total += count
            print(f"║  {name:<20} │ {count:>5} events sent{' ' * 22}║")
        print("╠══════════════════════════════════════════════════════════════╣")
        print(f"║  {'TOTAL':<20} │ {total:>5} events sent{' ' * 22}║")
        print("╚══════════════════════════════════════════════════════════════╝")
        print()
        print("  👉 View in Kafka UI: http://localhost:8080")
        print()


In [13]:
if __name__ == "__main__":
    main()



╔══════════════════════════════════════════════════════════════╗
║       OmniOC Multi-Topic Kafka Producer (PyCharm)           ║
║       Broker: localhost:9092                                 ║
╠══════════════════════════════════════════════════════════════╣
║  ✅ clickstream-events   │ 50 events                         ║
║  ✅ device-telemetry     │ 30 events                         ║
║  ✅ order-events         │ 20 events                         ║
║  ✅ dq-alerts            │ 10 events                         ║
║  💊 Poison pills: ON  │  🕐 Late arrivals: ON                    ║
╚══════════════════════════════════════════════════════════════╝
  Press Ctrl+C to stop at any time

  🔌 Connected to Kafka at localhost:9092

[Order Events      ] #   1 → order_cancelled        key=ORD-00008976          (19 left)
[DQ Alerts         ] #   1 → 13b7899f               key=silver.order_lines    (9 left)
[Clickstream       ] #   1 → click                  key=ac41d286-a9e8-4879-9  (49 left)
[Telemetry 